In [ ]:
"""
Trích xuất keyframe từ video của btc (dùng transnetv2).
Video -> scence -> keyframes.
Embed keyframe bằng CLIP và BEiT3.
So sánh cosine giữa các frame.
Chỉ giữ lại các frame có độ tương đồng thấp hơn 0.9
Code này sẽ được chạy trên gg colab.
"""

from pathlib import Path
import numpy as np
import torch
import cv2
import timm
from PIL import Image
from transnetv2_pytorch import TransNetV2
from transformers import CLIPProcessor, CLIPModel

PROJECT_ROOT = Path(__file__).resolve().parents[3]
VIDEO_DIR = str(PROJECT_ROOT / "data" / "??")
KEYFRAMES_EMBEDDING_DIR = PROJECT_ROOT / "data" / "??"  # numpy
KEYFRAMES_EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

# Dùng để debug, không chạy hết toàn bộ tập data
EXTRACTED_KEYFRAMES = PROJECT_ROOT / "data"
EXTRACTED_KEYFRAMES.mkdir(parents=True, exist_ok=True)

transnet_model = TransNetV2(device="auto")
state_dict = torch.load(
    "/content/transnetv2-pytorch-weights.pth",
    map_location=transnet_model.device,
)
transnet_model.load_state_dict(state_dict)
transnet_model.eval()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # chỉ bật autocast fp16 khi có GPU

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()  # type: ignore
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
beit3_model = timm.create_model("beit3_base_patch16_224", pretrained=True, num_classes=0).to(DEVICE).eval()
data_config = timm.data.resolve_model_data_config(beit3_model)  # type: ignore
transforms = timm.data.create_transform(**data_config, is_training=False)  # type: ignore

SIMILARITY_THRESHOLD = 0.9
FRAMES_PER_SCENE = 4
EMBED_BATCH_SIZE = 64  


def get_CLIP_embedding(images, batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    """
    Embed danh sách ảnh bằng CLIP theo batch lớn (thay vì gọi lẻ tẻ theo scene).
    """
    if len(images) == 0:
        return np.empty((0, clip_model.config.projection_dim), dtype=np.float32) # type: ignore

    all_embeddings = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size]
        inputs = clip_processor(images=batch, return_tensors="pt", padding=True).to(DEVICE)  # type: ignore
        with torch.inference_mode(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            embedding = clip_model.get_image_features(pixel_values=inputs["pixel_values"])
            embedding = torch.nn.functional.normalize(embedding, p=2, dim=-1) # type: ignore
        all_embeddings.append(embedding.float().cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)


def get_BEiT3_embedding(images, batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    """
    Embed danh sách ảnh bằng BEiT3 theo batch lớn.
    """
    if len(images) == 0:
        return np.empty((0, beit3_model.num_features), dtype=np.float32) # type: ignore

    all_embeddings = []
    for i in range(0, len(images), batch_size):
        batch_imgs = images[i:i + batch_size]
        batch = torch.stack([transforms(image) for image in batch_imgs]).to(DEVICE)
        with torch.inference_mode(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            embeddings = beit3_model(batch)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=-1)
        all_embeddings.append(embeddings.float().cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)


def extract_frames_by_indices(video_path, target_indices) -> dict:
    """
    Trích xuất frame theo danh sách index CHO TOÀN BỘ VIDEO trong 1 lần decode
    (chỉ mở VideoCapture 1 lần, seek tới frame đầu tiên rồi đọc tuần tự tới
    frame cuối cùng cần lấy). Trả về dict {index: PIL.Image}.
    """
    frame_map: dict = {}
    if len(target_indices) == 0:
        return frame_map

    target_indices = np.asarray(sorted(set(int(i) for i in target_indices)))
    target_set = set(target_indices.tolist())
    first_idx = int(target_indices[0])
    last_idx = int(target_indices[-1])

    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, first_idx)
    current_idx = first_idx

    while cap.isOpened() and current_idx <= last_idx:
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_set:
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_map[current_idx] = Image.fromarray(rgb_frame)

        current_idx += 1

    cap.release()
    return frame_map


def filter_frames(
    video_path,
    keyframe_dir,
    frames_per_scene: int = FRAMES_PER_SCENE,
    similarity_threshold: float = SIMILARITY_THRESHOLD,
    save_jpg: bool = False,
    embed_batch_size: int = EMBED_BATCH_SIZE,
):
    """
    Chuyển video -> scene
    Chuyển scene -> 4 keyframes
    Tính cosine similarity và lọc các frame.
    """
    video_name = video_path.stem
    scenes = transnet_model.detect_scenes(video_path, threshold=0.5)

    if not scenes:
        print(f"No scenes detected in {video_path}")
        return []

    # ---- 1) Gom index của TẤT CẢ scene lại trước ----
    scene_target_indices = []  # list[np.ndarray], song song với `scenes`
    all_indices_flat = []
    for scene in scenes:
        idxs = np.unique(
            np.linspace(scene["start_frame"], scene["end_frame"], frames_per_scene, dtype=int)
        )
        scene_target_indices.append(idxs)
        all_indices_flat.append(idxs)

    flat_unique_indices = np.unique(np.concatenate(all_indices_flat))

    # ---- 2) Decode video 1 LẦN cho toàn bộ index cần dùng ----
    frame_map = extract_frames_by_indices(video_path, flat_unique_indices)

    if not frame_map:
        print(f"No keyframes extracted from {video_path}")
        return []

    # ---- 3) Embed 1 LẦN theo batch lớn cho toàn bộ frame của video ----
    ordered_indices = sorted(frame_map.keys())
    ordered_frames = [frame_map[i] for i in ordered_indices]
    clip_all = get_CLIP_embedding(ordered_frames, batch_size=embed_batch_size)
    beit3_all = get_BEiT3_embedding(ordered_frames, batch_size=embed_batch_size)
    idx_to_pos = {idx: pos for pos, idx in enumerate(ordered_indices)}

    kept_clip_embeddings = []
    kept_beit3_embeddings = []
    kept_frame_names = []

    # cache ma trận đã giữ, chỉ rebuild khi có phần tử MỚI (tránh O(N^2))
    kept_clip_matrix = None
    kept_beit3_matrix = None

    def append_kept(clip_emb, beit3_emb, name):
        nonlocal kept_clip_matrix, kept_beit3_matrix
        kept_clip_embeddings.append(clip_emb)
        kept_beit3_embeddings.append(beit3_emb)
        kept_frame_names.append(name)
        kept_clip_matrix = np.stack(kept_clip_embeddings)
        kept_beit3_matrix = np.stack(kept_beit3_embeddings)

    for scene, target_indices in zip(scenes, scene_target_indices):
        # index nào không decode được (video ngắn hơn dự kiến) thì bỏ qua
        valid_positions = [(i, idx_to_pos[idx]) for i, idx in enumerate(target_indices) if idx in idx_to_pos]
        if not valid_positions:
            continue

        local_idx_list = [p[0] for p in valid_positions]   # vị trí trong target_indices (0..3)
        pos_list = [p[1] for p in valid_positions]          # vị trí trong mảng embedding toàn video

        clip_embeddings = clip_all[pos_list]
        beit3_embeddings = beit3_all[pos_list]
        extracted_frames = [ordered_frames[p] for p in pos_list]

        # Lấy frame ở giữa làm mốc, luôn giữ lại
        middle_local = len(local_idx_list) // 2
        middle_target_idx = target_indices[local_idx_list[middle_local]]

        frame_name = f"shot{scene['shot_id']}_frame{middle_target_idx}"
        if save_jpg:
            keyframes_save_path = keyframe_dir / video_name / f"{video_name}_{frame_name}.jpg"
            extracted_frames[middle_local].save(keyframes_save_path)

        append_kept(clip_embeddings[middle_local], beit3_embeddings[middle_local], frame_name)

        for pos, (frame, clip_emb, beit3_emb) in enumerate(
            zip(extracted_frames, clip_embeddings, beit3_embeddings)
        ):
            if pos == middle_local:
                continue

            max_clip_sim = float(np.max(kept_clip_matrix @ clip_emb))
            max_beit3_sim = float(np.max(kept_beit3_matrix @ beit3_emb))

            if max_clip_sim >= similarity_threshold and max_beit3_sim >= similarity_threshold:
                continue

            target_idx = target_indices[local_idx_list[pos]]
            frame_name = f"shot{scene['shot_id']}_frame{target_idx}"
            if save_jpg:
                keyframes_save_path = keyframe_dir / video_name / f"{video_name}_{frame_name}.jpg"
                frame.save(keyframes_save_path)

            append_kept(clip_emb, beit3_emb, frame_name)

    if not kept_clip_embeddings:
        print(f"No keyframes extracted from {video_path}")
        return []

    clip_array = np.stack(kept_clip_embeddings)      # shape: (N, clip_dim)
    beit3_array = np.stack(kept_beit3_embeddings)     # shape: (N, beit3_dim)
    np.save(KEYFRAMES_EMBEDDING_DIR / f"{video_name}_clip.npy", clip_array)
    np.save(KEYFRAMES_EMBEDDING_DIR / f"{video_name}_beit3.npy", beit3_array)
    np.save(KEYFRAMES_EMBEDDING_DIR / f"{video_name}_frame_names.npy", np.array(kept_frame_names))

    return kept_frame_names


if __name__ == "__main__":
    VIDEO_DIR = PROJECT_ROOT / "data" / "??"
    video_paths = sorted(VIDEO_DIR.glob("*.mp4"))

    for i, video_path in enumerate(video_paths):
        filter_frames(
            video_path=video_path,
            keyframe_dir=EXTRACTED_KEYFRAMES,
            save_jpg=(i < 3),  # chỉ lưu jpg cho 3 video đầu tiên
        )